In [1]:
!mkdir -p /kaggle/working/fmpriv && cp /kaggle/input/datasets/saharaskari1234/privacy-bias-fairnesss/*.py /kaggle/working/fmpriv/
%cd /kaggle/working/fmpriv
!ls
!python fmpriv.py --selftest 2>&1 | grep -E 'unit tests|SELFTEST|Traceback|Error'

/kaggle/working/fmpriv
fmpriv.py  prep.py
unit tests OK
SELFTEST OK; outputs in /tmp/tmpj_ffuac7


In [15]:
link = '/kaggle/working/data/CheXpert-v1.0-small'
if os.path.lexists(link): os.remove(link)
os.symlink(CX, link)
print(os.listdir(link)[:5])        # should show train, train.csv, valid, valid.csv

['valid.csv', 'valid', 'train.csv', 'train']


In [18]:
import os, sys; sys.path.insert(0, '/kaggle/working/fmpriv'); import prep
#WORK = '/kaggle/working'
#CX   = '/kaggle/input/datasets/ashery/chexpert'
DEMO = '/kaggle/input/datasets/saharaskari1234/privacy-bias-fairnesss/CHEXPERT DEMO.xlsx'   # fill in: !ls /kaggle/input/datasets/<user>

#os.makedirs('/kaggle/working/data', exist_ok=True)
#if not os.path.exists('/kaggle/working/data/CheXpert-v1.0-small'): os.symlink(CX, '/kaggle/working/data/CheXpert-v1.0-small')
ATTRS = ('sex', 'race', 'age_bin')
meta = prep.build_meta_chexpert('/kaggle/working/data', DEMO, out=f'{WORK}/meta.csv', max_patients=15000)
meta = meta[meta.path.map(os.path.exists)].reset_index(drop=True); meta.to_csv(f'{WORK}/meta.csv', index=False)
print(len(meta), 'images,', meta.patient_id.nunique(), 'patients')

/kaggle/working/meta.csv: 44622 images, 15000 patients; race: {'WHITE': 8395, 'OTHER': 2563, 'UNKNOWN': 1757, 'ASIAN': 1594, 'BLACK': 691}; use attrs=('sex','race','age_bin')
44622 images, 15000 patients


In [5]:
import os, sys; sys.path.insert(0, '/kaggle/working/fmpriv'); import prep
NIH = '/kaggle/input/datasets/nih-chest-xrays/data'     # must contain Data_Entry_2017.csv (check with !ls)
ATTRS_NIH = ('sex', 'age_bin')
meta = prep.build_meta_nih(NIH, out='/kaggle/working/meta_nih.csv', max_patients=15000)
meta = meta[meta.path.map(os.path.exists)].reset_index(drop=True); meta.to_csv('/kaggle/working/meta_nih.csv', index=False)
print(len(meta), 'images,', meta.patient_id.nunique(), 'patients')

/kaggle/working/meta_nih.csv: 54521 images, 15000 patients; use attrs=('sex','age_bin')
54521 images, 15000 patients


In [6]:
!pip install -q open_clip_torch

In [7]:
!sed -i 's/vit-base-patch16-224/vit_base_patch16_224/g' /kaggle/working/fmpriv/prep.py
!grep -n "BiomedCLIP" /kaggle/working/fmpriv/prep.py
import importlib, prep; importlib.reload(prep)

43:    """model: 'rad-dino' (microsoft/rad-dino, CLS) | 'biomedclip' (open_clip hf-hub BiomedCLIP, image embedding) | 'dummy' (test).
55:        net, _, pre = open_clip.create_model_and_transforms("hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"); net = net.to(dev).eval()


<module 'prep' from '/kaggle/working/fmpriv/prep.py'>

In [12]:
WORK = '/kaggle/working'

prep.extract_embeddings('/kaggle/working/meta_nih.csv', model='biomedclip', out=f'{WORK}/emb_nih_biomedclip.npy', bs=128)

open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  784MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

51328/54521
saved /kaggle/working/emb_nih_biomedclip.npy


'/kaggle/working/emb_nih_biomedclip.npy'

In [32]:
import fmpriv as F
HEADS = dict(linear=dict(head='linear', lr=3e-3, wd=1e-2, epochs=30), mlp=dict(head='mlp', hidden=512, lr=1e-3, wd=1e-2, epochs=30))
def go(model, head, **kw):
    return F.run(F.Cfg(name=f'{model}_{head}', out=f'{WORK}/outputs', meta=f'{WORK}/meta_nih.csv', emb=f'{WORK}/emb_{model}.npy',
                       n_models=200, attrs=ATTRS, **HEADS[head], **kw))
res = go('biomedclip', 'linear')

16:27:57 | CONFIG {'name': 'biomedclip_linear', 'out': '/kaggle/working/outputs', 'meta': '/kaggle/working/meta.csv', 'emb': '/kaggle/working/emb_biomedclip.npy', 'head': 'linear', 'hidden': 512, 'n_models': 200, 'epochs': 30, 'bs': 2048, 'lr': 0.003, 'wd': 0.01, 'seed': 0, 'val_frac': 0.1, 'test_frac': 0.2, 'max_audit_patients': None, 'attrs': ('sex', 'race', 'age_bin'), 'min_group': 100, 'tail_q': 99.0, 'n_boot': 500, 'min_pos': 20, 'n_util_models': 20, 'device': 'cuda'}
16:27:57 | ENV {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'pandas': '2.3.3', 'gpu': 'Tesla T4'}
16:27:57 | [start] load
16:27:57 | DATA records=44622 patients=15000 d=512 labels=14 records/patient=2.97
16:27:57 | SPLIT patients audit/val/test = 10500/1500/3000 ; records = 31601/4207/8814
16:27:58 | [done ] load (0.5s)
16:27:58 | [start] train 200 target heads (linear)
16:27:58 |   [targets] step 0/480 mean member-loss/model=0.8075
16:27:58 |   [targets] step 60/480 mean member-loss/model=0.5980


In [33]:
def go2(model, head, seed=0, **kw):
    name = f'{model}_{head}' + (f'_s{seed}' if seed else '')
    if os.path.exists(f'{WORK}/outputs/{name}/paper_numbers.json'):
        print('skip (already done):', name); return
    return F.run(F.Cfg(name=name, out=f'{WORK}/outputs', meta=f'{WORK}/meta.csv', emb=f'{WORK}/emb_{model}.npy',
                       n_models=200, attrs=ATTRS, seed=seed, **HEADS[head], **kw))

go2('biomedclip', 'mlp')                                   # capacity axis (seed 0)
for s in (1, 2):                                           # repeat with new splits and memberships
    go2('biomedclip', 'linear', seed=s)
    go2('biomedclip', 'mlp', seed=s)

print(F.compare_runs(f'{WORK}/outputs').to_string(index=False))

16:31:58 | CONFIG {'name': 'biomedclip_mlp', 'out': '/kaggle/working/outputs', 'meta': '/kaggle/working/meta.csv', 'emb': '/kaggle/working/emb_biomedclip.npy', 'head': 'mlp', 'hidden': 512, 'n_models': 200, 'epochs': 30, 'bs': 2048, 'lr': 0.001, 'wd': 0.01, 'seed': 0, 'val_frac': 0.1, 'test_frac': 0.2, 'max_audit_patients': None, 'attrs': ('sex', 'race', 'age_bin'), 'min_group': 100, 'tail_q': 99.0, 'n_boot': 500, 'min_pos': 20, 'n_util_models': 20, 'device': 'cuda'}
16:31:58 | ENV {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'pandas': '2.3.3', 'gpu': 'Tesla T4'}
16:31:58 | [start] load
16:31:58 | DATA records=44622 patients=15000 d=512 labels=14 records/patient=2.97
16:31:58 | SPLIT patients audit/val/test = 10500/1500/3000 ; records = 31601/4207/8814
16:31:58 | [done ] load (0.5s)
16:31:58 | [start] train 200 target heads (mlp)
16:31:59 |   [targets] step 0/480 mean member-loss/model=0.7423
16:32:13 |   [targets] step 60/480 mean member-loss/model=0.2995
16:32:26 

In [14]:
import fmpriv as F
HEADS = dict(linear=dict(head='linear', lr=3e-3, wd=1e-2, epochs=30), mlp=dict(head='mlp', hidden=512, lr=1e-3, wd=1e-2, epochs=30))
ON = f'{WORK}/outputs_nih'
def go_nih(model, head, seed=0, name=None, **kw):
    name = name or f'{model}_{head}' + (f'_s{seed}' if seed else '')
    if os.path.exists(f'{ON}/{name}/paper_numbers.json'): print('skip:', name); return
    return F.run(F.Cfg(name=name, out=ON, meta=f'{WORK}/meta_nih.csv', emb=f'{WORK}/emb_nih_{model}.npy',
                       n_models=200, attrs=('sex', 'age_bin'), seed=seed, **{**HEADS[head], **kw}))

for s in (0, 1, 2):
    go_nih('biomedclip', 'linear', seed=s); go_nih('biomedclip', 'mlp', seed=s)
go_nih('biomedclip', 'mlp', name='biomedclip_mlp_ep5', epochs=5)
print(F.compare_runs(ON).to_string(index=False))

19:59:24 | CONFIG {'name': 'biomedclip_linear', 'out': '/kaggle/working/outputs_nih', 'meta': '/kaggle/working/meta_nih.csv', 'emb': '/kaggle/working/emb_nih_biomedclip.npy', 'head': 'linear', 'hidden': 512, 'n_models': 200, 'epochs': 30, 'bs': 2048, 'lr': 0.003, 'wd': 0.01, 'seed': 0, 'val_frac': 0.1, 'test_frac': 0.2, 'max_audit_patients': None, 'attrs': ('sex', 'age_bin'), 'min_group': 100, 'tail_q': 99.0, 'n_boot': 500, 'min_pos': 20, 'n_util_models': 20, 'device': 'cuda'}
19:59:24 | ENV {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'pandas': '2.3.3', 'gpu': 'Tesla T4'}
19:59:24 | [start] load
19:59:25 | DATA records=54521 patients=15000 d=512 labels=15 records/patient=3.63
19:59:25 | SPLIT patients audit/val/test = 10500/1500/3000 ; records = 38316/5561/10644
19:59:25 | [done ] load (0.8s)
19:59:25 | [start] train 200 target heads (linear)
19:59:26 |   [targets] step 0/570 mean member-loss/model=0.8031
19:59:27 |   [targets] step 71/570 mean member-loss/model=0.

In [19]:
!cd /kaggle/working && zip -qr outputs.zip outputs_nih

In [16]:
import importlib, prep; importlib.reload(prep)
prep.extract_embeddings('/kaggle/working/meta_nih.csv', model='rad-dino', out=f'{WORK}/emb_nih_raddino.npy', bs=32)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

54432/54521
saved /kaggle/working/emb_nih_raddino.npy


'/kaggle/working/emb_nih_raddino.npy'

In [15]:
import importlib, prep; importlib.reload(prep)
prep.extract_embeddings(f'{WORK}/meta.csv', model='rad-dino',
                        out=f'{WORK}/emb_raddino.npy', bs=32)

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [36]:
for h in ['linear', 'mlp']:
    for s in [0, 1, 2]:
        go2('raddino', h, seed=s)
F.compare_runs(f'{WORK}/outputs')

17:28:03 | CONFIG {'name': 'raddino_linear', 'out': '/kaggle/working/outputs', 'meta': '/kaggle/working/meta.csv', 'emb': '/kaggle/working/emb_raddino.npy', 'head': 'linear', 'hidden': 512, 'n_models': 200, 'epochs': 30, 'bs': 2048, 'lr': 0.003, 'wd': 0.01, 'seed': 0, 'val_frac': 0.1, 'test_frac': 0.2, 'max_audit_patients': None, 'attrs': ('sex', 'race', 'age_bin'), 'min_group': 100, 'tail_q': 99.0, 'n_boot': 500, 'min_pos': 20, 'n_util_models': 20, 'device': 'cuda'}
17:28:03 | ENV {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'pandas': '2.3.3', 'gpu': 'Tesla T4'}
17:28:03 | [start] load
17:28:03 | DATA records=44622 patients=15000 d=768 labels=14 records/patient=2.97
17:28:03 | SPLIT patients audit/val/test = 10500/1500/3000 ; records = 31601/4207/8814
17:28:03 | [done ] load (0.5s)
17:28:03 | [start] train 200 target heads (linear)
17:28:03 |   [targets] step 0/480 mean member-loss/model=0.8057
17:28:04 |   [targets] step 60/480 mean member-loss/model=0.5612
17:28:

,run,test_auroc,agg_mia_auc,tpr_at_1e3,patient_p99,frac_ge_0_9,null_p99
0,biomedclip_linear,0.736482,0.568481,0.005113,0.723983,0.000286,0.609352
1,biomedclip_linear_s1,0.731014,0.567807,0.004324,0.724904,0.000286,0.606180
2,biomedclip_linear_s2,0.732160,0.568276,0.004583,0.726224,0.000381,0.607476
3,biomedclip_mlp,0.734185,0.697684,0.016745,0.940484,0.029333,0.604890
4,biomedclip_mlp_s1,0.732683,0.696626,0.015534,0.935469,0.028667,0.605699
5,biomedclip_mlp_s2,0.726596,0.698677,0.016394,0.941203,0.028762,0.606819
6,raddino_linear,0.778635,0.694387,0.030163,0.921973,0.014381,0.608250
7,raddino_linear_s1,0.769160,0.693057,0.028011,0.917491,0.013810,0.603985
8,raddino_linear_s2,0.767722,0.693369,0.028242,0.913239,0.012476,0.606169
9,raddino_mlp,0.770469,0.907178,0.126321,0.999487,0.592762,0.606120


In [17]:
for s in (0, 1, 2):
    go_nih('raddino', 'linear', seed=s); go_nih('raddino', 'mlp', seed=s)
go_nih('raddino', 'mlp', name='raddino_mlp_ep5', epochs=5)
print(F.compare_runs(ON).to_string(index=False))

20:34:07 | CONFIG {'name': 'raddino_linear', 'out': '/kaggle/working/outputs_nih', 'meta': '/kaggle/working/meta_nih.csv', 'emb': '/kaggle/working/emb_nih_raddino.npy', 'head': 'linear', 'hidden': 512, 'n_models': 200, 'epochs': 30, 'bs': 2048, 'lr': 0.003, 'wd': 0.01, 'seed': 0, 'val_frac': 0.1, 'test_frac': 0.2, 'max_audit_patients': None, 'attrs': ('sex', 'age_bin'), 'min_group': 100, 'tail_q': 99.0, 'n_boot': 500, 'min_pos': 20, 'n_util_models': 20, 'device': 'cuda'}
20:34:07 | ENV {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'pandas': '2.3.3', 'gpu': 'Tesla T4'}
20:34:07 | [start] load
20:34:07 | DATA records=54521 patients=15000 d=768 labels=15 records/patient=3.63
20:34:07 | SPLIT patients audit/val/test = 10500/1500/3000 ; records = 38316/5561/10644
20:34:08 | [done ] load (0.6s)
20:34:08 | [start] train 200 target heads (linear)
20:34:08 |   [targets] step 0/570 mean member-loss/model=0.8046
20:34:08 |   [targets] step 71/570 mean member-loss/model=0.5584
2

In [41]:
import pandas as pd, json, glob, re
rows = []
for p in glob.glob(f'{WORK}/outputs/*/paper_numbers.json'):
    n = p.split('/')[-2]
    cfg = re.match(r'(.+?_(?:linear|mlp))(?:_s\d+)?$', n).group(1)
    rows.append(dict(config=cfg, **pd.json_normalize(json.load(open(p))).iloc[0].to_dict()))
df = pd.DataFrame(rows)
num = df.select_dtypes('number').columns
agg = df.groupby('config')[num].agg(['mean','std']).round(3)
agg.to_csv(f'{WORK}/seed_aggregate.csv'); print(agg.shape);agg

(4, 62)


tail_threshold_auc         \
                                mean    std   
config                                        
biomedclip_linear              0.725  0.001   
biomedclip_mlp                 0.939  0.003   
raddino_linear                 0.918  0.004   
raddino_mlp                    1.000  0.000   

                  attack.closed_form_check.mean_abs_diff       \
                                                    mean  std   
config                                                          
biomedclip_linear                                  0.007  0.0   
biomedclip_mlp                                     0.006  0.0   
raddino_linear                                     0.006  0.0   
raddino_mlp                                        0.004  0.0   

                  attack.closed_form_check.corr        attack.aggregate.auc  \
                                           mean    std                 mean   
config                                                                        
biomedclip_linear                         0.992  0.001                0.568   
biomedclip_mlp                            0.996  0.001                0.698   
raddino_linear                            0.998  0.000                0.694   
raddino_mlp                               0.999  0.000                0.907   

                         attack.aggregate.n_pairs           ...  \
                     std                     mean      std  ...   
config                                                      ...   
biomedclip_linear  0.000              3999610.333  519.945  ...   
biomedclip_mlp     0.001              3999610.333  519.945  ...   
raddino_linear     0.001              3999610.333  519.945  ...   
raddino_mlp        0.000              3999610.333  519.945  ...   

                  n.audit_patients      n.val_patients      n.test_patients  \
                              mean  std           mean  std            mean   
config                                                                        
biomedclip_linear          10500.0  0.0         1500.0  0.0          3000.0   
biomedclip_mlp             10500.0  0.0         1500.0  0.0          3000.0   
raddino_linear             10500.0  0.0         1500.0  0.0          3000.0   
raddino_mlp                10500.0  0.0         1500.0  0.0          3000.0   

                       n.audit_records         n.models       
                   std            mean     std     mean  std  
config                                                        
biomedclip_linear  0.0       31384.333  188.32    200.0  0.0  
biomedclip_mlp     0.0       31384.333  188.32    200.0  0.0  
raddino_linear     0.0       31384.333  188.32    200.0  0.0  
raddino_mlp        0.0       31384.333  188.32    200.0  0.0  

[4 rows x 62 columns]

In [42]:
import numpy as np, pandas as pd, json, os, glob
from scipy import stats
import matplotlib.pyplot as plt
OUT = f'{WORK}/outputs'

def go3(model, name, **kw):                                   # custom-named run (skips if finished)
    if os.path.exists(f'{OUT}/{name}/paper_numbers.json'): print('skip', name); return
    F.run(F.Cfg(name=name, out=OUT, meta=f'{WORK}/meta.csv', emb=f'{WORK}/emb_{model}.npy', n_models=200, attrs=ATTRS, **kw))

# ============ #4 capacity sweep (embeddings are already standardised inside F.run) ============
for m in ['biomedclip', 'raddino']:
    for h in (8, 64): go3(m, f'{m}_mlp_h{h}', head='mlp', hidden=h, lr=1e-3, wd=1e-2, epochs=30)
    go3(m, f'{m}_mlp_ep5', head='mlp', hidden=512, lr=1e-3, wd=1e-2, epochs=5)        # early-stopping proxy
cmp = F.compare_runs(OUT); cmp['model'] = cmp.run.str.split('_').str[0]
cap = cmp[~cmp.run.str.contains('leace')].sort_values(['model', 'test_auroc'])
print('\n=== CAPACITY: leakage vs utility ===\n', cap.round(3).to_string(index=False)); cap.to_csv(f'{WORK}/capacity_table.csv', index=False)
plt.figure(figsize=(5, 4))
for m, g in cap.groupby('model'): plt.scatter(g.test_auroc, g.agg_mia_auc, label=m, s=40)
plt.axhline(.5, c='gray', ls=':'); plt.xlabel('test macro-AUROC'); plt.ylabel('aggregate MIA AUC'); plt.legend(); plt.tight_layout()
plt.savefig(f'{WORK}/fig_capacity.png', dpi=200)

# ============ #2 equivalence tests (95% CI inside +-delta  =>  TOST passes at alpha=.025) ============
DELTA, DR2 = 0.10, 0.01
A = pd.concat([pd.read_csv(p).assign(run=os.path.basename(os.path.dirname(p))) for p in sorted(glob.glob(f'{OUT}/*/patient_assoc.csv'))])
A['equiv_partial'] = (A.part_lo > -DELTA) & (A.part_hi < DELTA); A['equiv_dR2'] = A.dR2_hi < DR2
cols = ['run', 'attr', 'n', 'partial', 'part_lo', 'part_hi', 'equiv_partial', 'dR2_enc', 'dR2_hi', 'equiv_dR2']
print(f'\n=== EQUIVALENCE (|partial rho|<{DELTA}, dR2<{DR2}) ===\n', A[cols].round(3).to_string(index=False)); A[cols].to_csv(f'{WORK}/tost.csv', index=False)

# ============ #3 who leaks? rank-OLS of patient MIA AUC on candidate predictors ============
meta = pd.read_csv(f'{WORK}/meta.csv'); labs = [c for c in meta.columns if c not in F.NONLABEL]; Lb = meta[labs]; pid = meta.patient_id.astype(str)
ex = pd.DataFrame({'uncertain_frac': (Lb == -1).mean(1).groupby(pid).mean()})
rare = [l for l in labs if 0 < (Lb[l] == 1).mean() < 0.05]; ex['rare_findings'] = (Lb[rare] == 1).sum(1).groupby(pid).mean()
if 'Support Devices' in labs: ex['support_devices'] = (Lb['Support Devices'] == 1).groupby(pid).mean()
print('rare labels used:', rare)

def ols(df, y, xs):
    R = lambda v: stats.rankdata(v) / len(v); d = df[[y] + xs].dropna(); xs = [x for x in xs if d[x].nunique() > 1]
    Y = R(d[y].values); Xm = np.column_stack([np.ones(len(d))] + [(lambda v: (v - v.mean()) / v.std())(R(d[x].values)) for x in xs])
    b = np.linalg.pinv(Xm) @ Y; r = Y - Xm @ b; tot = ((Y - Y.mean()) ** 2).sum(); n, k = Xm.shape
    se = np.sqrt(np.diag(np.linalg.pinv(Xm.T @ Xm)) * (r @ r) / (n - k)); r2 = 1 - r @ r / tot; rows = []
    for j, x in enumerate(xs):
        keep = [i for i in range(k) if i != j + 1]; r_ = Y - Xm[:, keep] @ (np.linalg.pinv(Xm[:, keep]) @ Y)
        rows.append(dict(pred=x, beta=b[j + 1], lo=b[j + 1] - 1.96 * se[j + 1], hi=b[j + 1] + 1.96 * se[j + 1], dR2=r2 - (1 - r_ @ r_ / tot)))
    return n, r2, pd.DataFrame(rows)

W = []
for run in ['biomedclip_linear', 'biomedclip_mlp', 'raddino_linear', 'raddino_mlp']:
    d = pd.read_csv(f'{OUT}/{run}/patient_level.csv.gz'); d = d.join(ex, on=d.patient_id.astype(str))
    d['enc_mean'] = d[[c for c in d if c.startswith('enc_')]].mean(1)
    n, r2, t = ols(d, 'mia', ['log_nrec', 'sparsity', 'n_findings', 'uncertain_frac', 'rare_findings', 'support_devices', 'age', 'enc_mean'])
    print(f'\n=== WHO LEAKS: {run} (n={n}, R2={r2:.3f}) ===\n', t.round(3).to_string(index=False)); W.append(t.assign(run=run, R2=r2))
pd.concat(W).to_csv(f'{WORK}/who_leaks.csv', index=False)

# ============ LEACE: erase sex/race/age from embeddings, then re-audit ============
def leace_fit(X, Z):                                          # X (n,d), Z (n,c) concept one-hot; returns mean, erasure matrix
    mu = X.mean(0); Xc = X - mu; Zc = Z - Z.mean(0); Sxx = Xc.T @ Xc / len(X); Sxz = Xc.T @ Zc / len(X)
    ev, U = np.linalg.eigh(Sxx); ev = np.clip(ev, 1e-6 * ev.max(), None); Wh = (U / np.sqrt(ev)) @ U.T; Wp = (U * np.sqrt(ev)) @ U.T
    u, s, _ = np.linalg.svd(Wh @ Sxz, full_matrices=False); u = u[:, s > 1e-6 * s.max()]
    return mu, Wp @ u @ u.T @ Wh

age_bin = pd.cut(meta.age, [0, 40, 60, 80, 500], labels=['<40', '40-60', '60-80', '80+'], right=False).astype(str)
Z = pd.concat([pd.get_dummies(meta[c].astype(str)) for c in ('sex', 'race')] + [pd.get_dummies(age_bin)], axis=1).to_numpy(np.float64)
for m in ['biomedclip', 'raddino']:
    fn = f'{WORK}/emb_{m}_leace.npy'
    if not os.path.exists(fn):
        X = np.load(f'{WORK}/emb_{m}.npy').astype(np.float64); mu, Pm = leace_fit(X, Z); Xe = X - (X - mu) @ Pm.T
        print(m, 'LEACE: max |cross-cov(x_erased, z)| =', np.abs((Xe - Xe.mean(0)).T @ (Z - Z.mean(0)) / len(X)).max().round(8)); np.save(fn, Xe.astype(np.float32))
    for h in ('linear', 'mlp'): go2(f'{m}_leace', h)
cmp = F.compare_runs(OUT); print('\n=== LEACE vs ORIGINAL ===\n', cmp[cmp.run.str.match(r'^(biomedclip|raddino)(_leace)?_(linear|mlp)$')].round(3).to_string(index=False))
enc = pd.concat([pd.read_csv(f'{OUT}/{r}/encoding_overall.csv').assign(run=r) for r in
                 ['biomedclip_linear', 'biomedclip_leace_linear', 'raddino_linear', 'raddino_leace_linear']])
print('\n=== DEMOGRAPHIC ENCODING after erasure (macro AUROC; chance = 0.5 for binary) ===\n', enc.pivot(index='run', columns='attr', values='macro_auroc').round(3))
fr = pd.concat([pd.read_csv(f'{OUT}/{r}/fairness_table.csv').assign(run=r) for r in ['biomedclip_mlp', 'biomedclip_leace_mlp', 'raddino_mlp', 'raddino_leace_mlp']])
print('\n=== FAIRNESS after erasure (underdiagnosis gap) ===\n', fr.pivot_table(index=['attr', 'group'], columns='run', values='ud_gap').round(3))

17:49:16 | CONFIG {'name': 'biomedclip_mlp_h8', 'out': '/kaggle/working/outputs', 'meta': '/kaggle/working/meta.csv', 'emb': '/kaggle/working/emb_biomedclip.npy', 'head': 'mlp', 'hidden': 8, 'n_models': 200, 'epochs': 30, 'bs': 2048, 'lr': 0.001, 'wd': 0.01, 'seed': 0, 'val_frac': 0.1, 'test_frac': 0.2, 'max_audit_patients': None, 'attrs': ('sex', 'race', 'age_bin'), 'min_group': 100, 'tail_q': 99.0, 'n_boot': 500, 'min_pos': 20, 'n_util_models': 20, 'device': 'cuda'}
17:49:16 | ENV {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'pandas': '2.3.3', 'gpu': 'Tesla T4'}
17:49:16 | [start] load
17:49:17 | DATA records=44622 patients=15000 d=512 labels=14 records/patient=2.97
17:49:17 | SPLIT patients audit/val/test = 10500/1500/3000 ; records = 31601/4207/8814
17:49:17 | [done ] load (0.4s)
17:49:17 | [start] train 200 target heads (mlp)
17:49:17 |   [targets] step 0/480 mean member-loss/model=0.7420
17:49:17 |   [targets] step 60/480 mean member-loss/model=0.5578
17:49:18

In [43]:
import pandas as pd, os
OUT = f'{WORK}/outputs'
ok = lambda r: os.path.exists(f'{OUT}/{r}/paper_numbers.json')
print('missing runs:', [r for r in ['biomedclip_leace_linear','biomedclip_leace_mlp','raddino_leace_linear','raddino_leace_mlp'] if not ok(r)])

cmp = pd.read_csv(f'{OUT}/compare_runs.csv')
print('\n=== MIA / utility ===')
print(cmp[cmp.run.str.match(r'^(biomedclip|raddino)(_leace)?_(linear|mlp)$')].round(3).to_string(index=False))

print('\n=== ENCODING macro-AUROC (0.5 = chance for binary) ===')
runs = [r for r in ['biomedclip_linear','biomedclip_leace_linear','raddino_linear','raddino_leace_linear'] if ok(r)]
enc = pd.concat([pd.read_csv(f'{OUT}/{r}/encoding_overall.csv').assign(run=r) for r in runs])
print(enc.pivot(index='run', columns='attr', values='macro_auroc').round(3))

print('\n=== FAIRNESS: underdiagnosis gap (ud_gap) and FNR gap, MLP heads ===')
runs = [r for r in ['biomedclip_mlp','biomedclip_leace_mlp','raddino_mlp','raddino_leace_mlp'] if ok(r)]
fr = pd.concat([pd.read_csv(f'{OUT}/{r}/fairness_table.csv').assign(run=r) for r in runs])
for v in ('ud_gap', 'fnr_gap'):
    print(f'-- {v}'); print(fr.pivot_table(index=['attr','group'], columns='run', values=v).round(3))

missing runs: []

=== MIA / utility ===
                    run  test_auroc  agg_mia_auc  tpr_at_1e3  patient_p99  frac_ge_0_9  null_p99
biomedclip_leace_linear       0.716        0.568       0.005        0.728        0.000     0.610
   biomedclip_leace_mlp       0.715        0.700       0.016        0.938        0.028     0.607
      biomedclip_linear       0.736        0.568       0.005        0.724        0.000     0.609
         biomedclip_mlp       0.734        0.698       0.017        0.940        0.029     0.605
   raddino_leace_linear       0.765        0.695       0.031        0.922        0.015     0.607
      raddino_leace_mlp       0.767        0.913       0.134        1.000        0.613     0.605
         raddino_linear       0.779        0.694       0.030        0.922        0.014     0.608
            raddino_mlp       0.770        0.907       0.126        0.999        0.593     0.606

=== ENCODING macro-AUROC (0.5 = chance for binary) ===
attr                     age_bi

In [18]:
import pandas as pd, glob, os
A = pd.concat([pd.read_csv(p).assign(run=os.path.basename(os.path.dirname(p))) for p in sorted(glob.glob(f'{ON}/*/patient_assoc.csv'))])
A['equiv_partial'] = (A.part_lo > -0.1) & (A.part_hi < 0.1); A['equiv_dR2'] = A.dR2_hi < 0.01
print(A[['run','attr','n','partial','part_lo','part_hi','equiv_partial','dR2_enc','dR2_hi','equiv_dR2']].round(3).to_string(index=False))
print(F.compare_runs(ON).round(3).to_string(index=False))

                 run    attr     n  partial  part_lo  part_hi  equiv_partial  dR2_enc  dR2_hi  equiv_dR2
   biomedclip_linear     sex 10500   -0.012   -0.032    0.007           True    0.000   0.001       True
   biomedclip_linear age_bin 10499   -0.038   -0.058   -0.020           True    0.001   0.002       True
biomedclip_linear_s1     sex 10500   -0.024   -0.043   -0.005           True    0.000   0.001       True
biomedclip_linear_s1 age_bin 10499   -0.050   -0.069   -0.033           True    0.002   0.003       True
biomedclip_linear_s2     sex 10500   -0.018   -0.039    0.001           True    0.000   0.001       True
biomedclip_linear_s2 age_bin 10499   -0.034   -0.051   -0.014           True    0.001   0.002       True
      biomedclip_mlp     sex 10500   -0.034   -0.054   -0.014           True    0.001   0.001       True
      biomedclip_mlp age_bin 10499   -0.119   -0.137   -0.099          False    0.006   0.008       True
  biomedclip_mlp_ep5     sex 10500   -0.007   -0.025   

In [44]:
import numpy as np, pandas as pd, torch, os, scipy.stats as sst, torch.nn as nn, torch.nn.functional as TF
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
OUT = f'{WORK}/outputs'; NM = globals().get('NM', 200); dev = 'cuda' if torch.cuda.is_available() else 'cpu'
HD = dict(linear=dict(head='linear', lr=3e-3, wd=1e-2, epochs=30), mlp=dict(head='mlp', hidden=512, lr=1e-3, wd=1e-2, epochs=30))
def R(model, head, seed=0, name=None):                         # same naming as go2; skips finished runs
    name = name or f'{model}_{head}' + (f'_s{seed}' if seed else '')
    if not os.path.exists(f'{OUT}/{name}/paper_numbers.json'):
        F.run(F.Cfg(name=name, out=OUT, meta=f'{WORK}/meta.csv', emb=f'{WORK}/emb_{model}.npy', n_models=NM, attrs=ATTRS, seed=seed, **HD[head]))
    return name
MODELS = ['biomedclip', 'raddino']

# ===== (1) fairness: original vs LEACE, MLP heads, 3 seeds =====
for m in MODELS:
    for s in (0, 1, 2): R(m, 'mlp', s); R(f'{m}_leace', 'mlp', s)
sfx = lambda s: f'_s{s}' if s else ''
for m in MODELS:
    ld = lambda p: pd.concat([pd.read_csv(f'{OUT}/{p}_mlp{sfx(s)}/fairness_table.csv').assign(seed=s) for s in (0, 1, 2)])
    d = ld(m).merge(ld(f'{m}_leace'), on=['attr', 'group', 'seed'], suffixes=('_o', '_l')); d['diff'] = d.ud_gap_o - d.ud_gap_l
    d['o_sig'] = (d.ud_gap_lo_o > 0) | (d.ud_gap_hi_o < 0)
    t = d.groupby(['attr', 'group']).agg(n=('seed', 'size'), gap_orig=('ud_gap_o', 'mean'), sd_o=('ud_gap_o', 'std'), gap_leace=('ud_gap_l', 'mean'), sd_l=('ud_gap_l', 'std'),
                                         diff=('diff', 'mean'), diff_sd=('diff', 'std'), seeds_diff_pos=('diff', lambda x: int((x > 0).sum())), seeds_orig_CI_excl0=('o_sig', 'sum')).reset_index()
    t = t[(t.gap_orig.abs() > 0) | (t.sd_o > 0)]; h = sst.t.ppf(.975, t.n - 1) * t.diff_sd / np.sqrt(t.n); t['diff_lo'], t['diff_hi'] = t['diff'] - h, t['diff'] + h
    print(f'\n=== (1) {m}: underdiagnosis gap, original vs LEACE, mean over 3 seeds (diff CI = crude t-interval, df=2) ===\n', t.round(3).to_string(index=False))
    t.to_csv(f'{WORK}/fair_seeds_{m}.csv', index=False)

# ===== (3) leakage scores that do not grow with #images: mean over records, one random record =====
_orig = F.patient_max; ST = {}
def _pm(th, pidx, Pa):
    out = _orig(th, pidx, Pa)
    if 'done' not in ST:                                       # first call per run = real membership (second = null)
        t = th.detach().cpu().numpy(); p = pidx.cpu().numpy(); ST['mean'] = np.bincount(p, weights=t, minlength=Pa) / np.maximum(np.bincount(p, minlength=Pa), 1)
        o = np.random.default_rng(0).permutation(len(p)); u, i = np.unique(p[o], return_index=True); one = np.full(Pa, np.nan); one[u] = t[o][i]; ST['one'] = one; ST['done'] = 1
    return out
F.patient_max = _pm
try:
    for m in MODELS:
        for hd in ('linear', 'mlp'):
            name = f'{m}_{hd}_alt'; pf = f'{OUT}/{name}/patient_level.csv.gz'
            if os.path.exists(pf) and 'mia_mean' in pd.read_csv(pf, nrows=1).columns: continue
            if os.path.exists(f'{OUT}/{name}/paper_numbers.json'): os.remove(f'{OUT}/{name}/paper_numbers.json')
            ST.clear(); R(m, hd, 0, name=name); d = pd.read_csv(pf); d['mia_mean'], d['mia_one'] = ST['mean'], ST['one']; d.to_csv(pf, index=False)
finally: F.patient_max = _orig
cmp = F.compare_runs(OUT).set_index('run'); print('\n=== (3) determinism check: agg MIA AUC original vs re-run ===')
print(pd.DataFrame({r: [cmp.loc[r, 'agg_mia_auc'] if r in cmp.index else np.nan, cmp.loc[r + '_alt', 'agg_mia_auc']] for r in [f'{m}_{h}' for m in MODELS for h in ('linear', 'mlp')]}, index=['orig', 'rerun']).round(3))

meta = pd.read_csv(f'{WORK}/meta.csv'); labs = [c for c in meta.columns if c not in F.NONLABEL]; Lb = meta[labs]; pid = meta.patient_id.astype(str)
ex = pd.DataFrame({'uncertain_frac': (Lb == -1).mean(1).groupby(pid).mean()}); rare = [l for l in labs if 0 < (Lb[l] == 1).mean() < 0.05]
ex['rare_findings'] = (Lb[rare] == 1).sum(1).groupby(pid).mean()
if 'Support Devices' in labs: ex['support_devices'] = (Lb['Support Devices'] == 1).groupby(pid).mean()
def ols(df, y, xs):
    Rk = lambda v: sst.rankdata(v) / len(v); d = df[[y] + xs].dropna(); xs = [x for x in xs if d[x].nunique() > 1]
    Y = Rk(d[y].values); Xm = np.column_stack([np.ones(len(d))] + [(lambda v: (v - v.mean()) / v.std())(Rk(d[x].values)) for x in xs])
    b = np.linalg.pinv(Xm) @ Y; r = Y - Xm @ b; tot = ((Y - Y.mean()) ** 2).sum(); r2 = 1 - r @ r / tot; rows = {}
    for j, x in enumerate(xs):
        keep = [i for i in range(Xm.shape[1]) if i != j + 1]; r_ = Y - Xm[:, keep] @ (np.linalg.pinv(Xm[:, keep]) @ Y); rows[x] = r2 - (1 - r_ @ r_ / tot)
    return r2, rows
PR = ['log_nrec', 'sparsity', 'n_findings', 'uncertain_frac', 'rare_findings', 'support_devices', 'age', 'enc_mean']; res = []
for m in MODELS:
    for hd in ('linear', 'mlp'):
        d = pd.read_csv(f'{OUT}/{m}_{hd}_alt/patient_level.csv.gz'); d = d.join(ex, on=d.patient_id.astype(str)); d['enc_mean'] = d[[c for c in d if c.startswith('enc_')]].mean(1)
        for y in ('mia', 'mia_mean', 'mia_one'):
            r2, rows = ols(d, y, PR); res.append(dict(run=f'{m}_{hd}', score=y, R2=r2, **rows))
res = pd.DataFrame(res); print('\n=== (3) delta-R2 by predictor: mia (max over records) vs mia_mean vs mia_one (single random record) ===\n', res.round(3).to_string(index=False)); res.to_csv(f'{WORK}/who_leaks_alt.csv', index=False)

# ===== (2) nonlinear probe: can demographics still be read after LEACE? =====
pc, pu = pd.factorize(meta.patient_id.astype(str)); P = len(pu); g = meta.groupby(pc)
pt = pd.DataFrame({'sex': g.sex.first().astype(str).str.upper(), 'race': g.race.first().astype(str).str.upper(), 'age': g.age.mean()})
pt['age_bin'] = pd.cut(pt.age, [0, 40, 60, 80, 500], labels=['<40', '40-60', '60-80', '80+'], right=False).astype(str)
def probe(Z, y, C, hidden, seed=0):
    Zt, yt = torch.tensor(Z, device=dev), torch.tensor(y, device=dev); post = np.zeros((len(y), C), np.float32)
    for tr, te in KFold(5, shuffle=True, random_state=seed).split(Z):
        torch.manual_seed(seed); d_ = Z.shape[1]; net = (nn.Linear(d_, C) if hidden == 0 else nn.Sequential(nn.Linear(d_, hidden), nn.ReLU(), nn.Linear(hidden, C))).to(dev)
        w = torch.tensor(len(tr) / (C * np.maximum(np.bincount(y[tr], minlength=C), 1)), dtype=torch.float32, device=dev)
        opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-2); tri = torch.tensor(tr, device=dev)
        for _ in range(20):
            pr = tri[torch.randperm(len(tri), device=dev)]
            for i in range(0, len(pr), 512):
                b = pr[i:i + 512]; loss = TF.cross_entropy(net(Zt[b]), yt[b], weight=w); opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): post[te] = torch.softmax(net(Zt[te]), 1).cpu().numpy()
    return roc_auc_score(y, post[:, 1]) if C == 2 else roc_auc_score(y, post, multi_class='ovr', average='macro')
rows = []
for m in MODELS:
    for tag, fn in (('original', f'emb_{m}.npy'), ('LEACE', f'emb_{m}_leace.npy')):
        X = np.load(f'{WORK}/{fn}').astype(np.float32); X = (X - X.mean(0)) / (X.std(0) + 1e-6)
        Zp = pd.DataFrame(X).groupby(pc).mean().to_numpy(np.float32); Zp = (Zp - Zp.mean(0)) / (Zp.std(0) + 1e-6)
        for a in ATTRS:
            ok = ~pt[a].isin(F.EXCL_PROBE).to_numpy(); cl, y = np.unique(pt[a].to_numpy()[ok], return_inverse=True)
            for hd, nm in ((0, 'linear'), (256, 'mlp')): rows.append(dict(model=m, emb=tag, probe=nm, attr=a, auroc=probe(Zp[ok], y, len(cl), hd)))
pr = pd.DataFrame(rows); print('\n=== (2) probe macro-AUROC on patient-mean embeddings (0.5 = chance for binary) ===')
print(pr.pivot_table(index=['model', 'emb', 'probe'], columns='attr', values='auroc').round(3)); pr.to_csv(f'{WORK}/probe_after_leace.csv', index=False)

18:42:30 | CONFIG {'name': 'biomedclip_leace_mlp_s1', 'out': '/kaggle/working/outputs', 'meta': '/kaggle/working/meta.csv', 'emb': '/kaggle/working/emb_biomedclip_leace.npy', 'head': 'mlp', 'hidden': 512, 'n_models': 200, 'epochs': 30, 'bs': 2048, 'lr': 0.001, 'wd': 0.01, 'seed': 1, 'val_frac': 0.1, 'test_frac': 0.2, 'max_audit_patients': None, 'attrs': ('sex', 'race', 'age_bin'), 'min_group': 100, 'tail_q': 99.0, 'n_boot': 500, 'min_pos': 20, 'n_util_models': 20, 'device': 'cuda'}
18:42:30 | ENV {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'pandas': '2.3.3', 'gpu': 'Tesla T4'}
18:42:30 | [start] load
18:42:30 | DATA records=44622 patients=15000 d=512 labels=14 records/patient=2.97
18:42:30 | SPLIT patients audit/val/test = 10500/1500/3000 ; records = 31292/4480/8850
18:42:31 | [done ] load (0.4s)
18:42:31 | [start] train 200 target heads (mlp)
18:42:31 |   [targets] step 0/480 mean member-loss/model=0.7431
18:42:44 |   [targets] step 60/480 mean member-loss/model=0

In [45]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
for m in ['biomedclip', 'raddino']:
    print(f'\n--- (1) fairness {m} ---'); print(pd.read_csv(f'{WORK}/fair_seeds_{m}.csv').round(3).to_string(index=False))
c = pd.read_csv(f'{WORK}/outputs/compare_runs.csv').set_index('run')
print('\n--- (3) determinism: agg MIA orig vs rerun ---')
print(pd.DataFrame({r: [c.agg_mia_auc.get(r), c.agg_mia_auc.get(r + '_alt')] for r in ['biomedclip_linear', 'biomedclip_mlp', 'raddino_linear', 'raddino_mlp']}, index=['orig', 'rerun']).round(3))
print('\n--- (3) who leaks: dR2 by score ---'); print(pd.read_csv(f'{WORK}/who_leaks_alt.csv').round(3).to_string(index=False))
print('\n--- (2) probes after LEACE ---')
print(pd.read_csv(f'{WORK}/probe_after_leace.csv').pivot_table(index=['model', 'emb', 'probe'], columns='attr', values='auroc').round(3))


--- (1) fairness biomedclip ---
   attr group  n  gap_orig  sd_o  gap_leace  sd_l   diff  diff_sd  seeds_diff_pos  seeds_orig_CI_excl0  diff_lo  diff_hi
age_bin 40-60  3     0.086 0.009      0.035 0.006  0.052    0.006               3                    3    0.037    0.066
age_bin   80+  3    -0.027 0.025      0.035 0.034 -0.062    0.014               0                    2   -0.096   -0.028
age_bin   <40  3     0.167 0.031      0.054 0.033  0.113    0.016               3                    3    0.073    0.153
   race ASIAN  3    -0.006 0.016     -0.008 0.010  0.003    0.006               2                    0   -0.013    0.018
   race BLACK  3     0.095 0.048      0.044 0.011  0.051    0.039               3                    2   -0.046    0.148
    sex     F  3    -0.009 0.011      0.001 0.010 -0.010    0.009               0                    0   -0.032    0.013

--- (1) fairness raddino ---
   attr group  n  gap_orig  sd_o  gap_leace  sd_l   diff  diff_sd  seeds_diff_pos  seeds_o

In [48]:
!cd /kaggle/working && zip -qr outputs.zip outputs

In [47]:
import subprocess, importlib, sys
subprocess.run("cd /kaggle/working/fmpriv && sed 's#microsoft/rad-dino#facebook/dinov2-base#g' prep.py > prep_dino.py", shell=True, check=True)
import prep_dino; importlib.reload(prep_dino)
prep_dino.extract_embeddings(f'{WORK}/meta.csv', model='rad-dino', out=f'{WORK}/emb_dinov2.npy', bs=64)   # 'rad-dino' branch now loads DINOv2 (CLS, 768-d)

for h in ['linear', 'mlp']:
    for s in [0, 1, 2]: go2('dinov2', h, seed=s)
F.compare_runs(f'{WORK}/outputs')

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

41664/44622
saved /kaggle/working/emb_dinov2.npy
19:13:38 | CONFIG {'name': 'dinov2_linear', 'out': '/kaggle/working/outputs', 'meta': '/kaggle/working/meta.csv', 'emb': '/kaggle/working/emb_dinov2.npy', 'head': 'linear', 'hidden': 512, 'n_models': 200, 'epochs': 30, 'bs': 2048, 'lr': 0.003, 'wd': 0.01, 'seed': 0, 'val_frac': 0.1, 'test_frac': 0.2, 'max_audit_patients': None, 'attrs': ('sex', 'race', 'age_bin'), 'min_group': 100, 'tail_q': 99.0, 'n_boot': 500, 'min_pos': 20, 'n_util_models': 20, 'device': 'cuda'}
19:13:38 | ENV {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'pandas': '2.3.3', 'gpu': 'Tesla T4'}
19:13:38 | [start] load
19:13:39 | DATA records=44622 patients=15000 d=768 labels=14 records/patient=2.97
19:13:39 | SPLIT patients audit/val/test = 10500/1500/3000 ; records = 31601/4207/8814
19:13:39 | [done ] load (0.6s)
19:13:39 | [start] train 200 target heads (linear)
19:13:39 |   [targets] step 0/480 mean member-loss/model=0.8046
19:13:40 |   [targets] s

,run,test_auroc,agg_mia_auc,tpr_at_1e3,patient_p99,frac_ge_0_9,null_p99
0,biomedclip_leace_linear,0.716022,0.567803,0.004900,0.728416,0.000286,0.610465
1,biomedclip_leace_mlp,0.715214,0.700476,0.016094,0.937724,0.028190,0.607406
2,biomedclip_leace_mlp_s1,0.710912,0.699045,0.014813,0.933594,0.027524,0.608216
3,biomedclip_leace_mlp_s2,0.708078,0.701526,0.015266,0.938235,0.030381,0.606506
4,biomedclip_linear,0.736482,0.568481,0.005113,0.723983,0.000286,0.609352
5,biomedclip_linear_alt,0.736482,0.568481,0.005113,0.723983,0.000286,0.609352
6,biomedclip_linear_s1,0.731014,0.567807,0.004324,0.724904,0.000286,0.606180
7,biomedclip_linear_s2,0.732160,0.568276,0.004583,0.726224,0.000381,0.607476
8,biomedclip_mlp,0.734185,0.697684,0.016745,0.940484,0.029333,0.604890
9,biomedclip_mlp_alt,0.734185,0.697684,0.016745,0.940484,0.029333,0.604890
